# 05_06 Sensitivity Noise Sweep

This sweep studies what happens when sensitive items are not only dropout-prone, but also produce noisier responses.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "_sweep_helpers.py").exists():
    sys.path.append(str(NOTEBOOK_DIR))
elif (NOTEBOOK_DIR / "notebooks" / "_sweep_helpers.py").exists():
    sys.path.append(str(NOTEBOOK_DIR / "notebooks"))
else:
    sys.path.append(str(Path(r"C:/Users/49160/Adaptive-Onboarding/notebooks")))

from _sweep_helpers import (
    POLICY_STYLE,
    config_label,
    equalize_y_axes,
    load_sweep,
    plot_dimension_group_lines,
    plot_metric_lines,
    plot_pareto,
    plot_weighted_delta,
    save,
    setup,
    show_sweeps,
)

setup()

## Select Sweep

In [ ]:
PARAM = "sensitivity-noise"
SWEEP_FILE = None  # e.g. "20260429_102545_sweep_sensitivity_noise.json"
CONFIG_FILTER = None

show_sweeps(PARAM)
sweep = load_sweep(PARAM, sweep_file=SWEEP_FILE, config_filter=CONFIG_FILTER)
print(config_label(sweep))

## Information Quality

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
plot_metric_lines(
    axes[0], sweep, "mean_final_d_error",
    xlabel="Sensitivity noise scale",
    ylabel="Mean D-error",
    title="All episodes",
)
plot_metric_lines(
    axes[1], sweep, "mean_final_d_error_completed",
    xlabel="Sensitivity noise scale",
    ylabel="Mean D-error",
    title="Completed only",
)
equalize_y_axes(axes[0], axes[1])
plot_metric_lines(
    axes[2], sweep, "mean_estimation_error",
    xlabel="Sensitivity noise scale",
    ylabel="Mean estimation error (lower is better)",
    title="Trait estimation",
)
fig.suptitle(config_label(sweep), y=1.03, fontsize=9)
plt.tight_layout()
# save(fig, "fig_05_06_sensitivity_noise_information")

## Trait-Group Uncertainty

For axis-targeted sensitivity configs, this splits posterior marginal variance into sensitive versus other traits as sensitive-response noise changes.

In [ ]:
metric = "mean_final_d_error_by_dimension"
comparison_policies = [p for p in ["surrogate_unweighted", "surrogate_weighted"] if p in sweep["conditions"][0]["policies"]]

if metric not in next(iter(sweep["conditions"][-1]["policies"].values())):
    print("This sweep was generated before per-dimension D-error was saved. Re-run experiments.sweep to populate it.")
elif not sweep.get("fixed_config", {}).get("sensitive_axes"):
    print("No sensitive_axes configured for this sweep; this grouped trait view is most useful for axis-targeted sensitivity.")
else:
    fig, axes = plt.subplots(1, len(comparison_policies), figsize=(5.8 * len(comparison_policies), 4), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, policy in zip(axes, comparison_policies):
        plot_dimension_group_lines(
            ax,
            sweep,
            metric,
            policy=policy,
            xlabel="Sensitivity noise scale",
            ylabel="Mean posterior variance" if ax is axes[0] else "",
            title=POLICY_STYLE[policy]["label"],
        )
    fig.suptitle(config_label(sweep), y=1.03, fontsize=9)
    plt.tight_layout()
    # save(fig, "fig_05_06_trait_group_uncertainty")
    plt.show()

## Completion And Exposure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
plot_metric_lines(
    axes[0], sweep, "dropout_rate",
    y_scale=100,
    xlabel="Sensitivity noise scale",
    ylabel="Episode dropout rate (%)",
    title="Dropout",
)
plot_metric_lines(
    axes[1], sweep, "sensitive_rate",
    y_scale=100,
    xlabel="Sensitivity noise scale",
    ylabel="Sensitive questions (%)",
    title="Sensitive exposure",
)
plot_metric_lines(
    axes[2], sweep, "mean_n_answered",
    xlabel="Sensitivity noise scale",
    ylabel="Mean questions answered",
    title="Completion length",
)
fig.suptitle(config_label(sweep), y=1.03, fontsize=9)
plt.tight_layout()
# save(fig, "fig_05_06_sensitivity_noise_completion")

## Tradeoff

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4))
plot_pareto(
    axes[0], sweep,
    y_metric="mean_estimation_error",
    ylabel="Mean estimation error (lower is better)",
    title="Dropout-risk vs error",
)
plot_weighted_delta(
    axes[1], sweep, "dropout_rate",
    y_scale=100,
    xlabel="Sensitivity noise scale",
    ylabel="Weighted - unweighted (pp)",
    title="Dropout-rate delta",
)
plot_weighted_delta(
    axes[2], sweep, "mean_estimation_error",
    xlabel="Sensitivity noise scale",
    ylabel="Weighted - unweighted",
    title="Estimation-error delta",
)
fig.suptitle(config_label(sweep), y=1.03, fontsize=9)
plt.tight_layout()
# save(fig, "fig_05_06_sensitivity_noise_tradeoff")